In [277]:
R.<x> = PolynomialRing(QQ)

K = NumberField(x^5 - 3*x^3 + 9*x - 8 , 'a')
n = K.degree()
a = K.gen()
B = [1, 1/2*a^4 + 1/2*a, 1/2*a^4 + 1/2*a^3 + 1/2*a^2, a^3, a^4]  # Basis for the ring of integers

In [6]:
# Writes a list of elements in the number field of degree n as a matrix, where the ith column represents the 
# ith element with respect to the power basis 1, a, a^2, ..., a^(n-1). 

def elems_to_matrix_power (l, n):
    M = [x.list() + [0] * (n - len(x.list())) for x in l]
    return Matrix(M).transpose()

In [7]:
# Represents elements as a matrix where 
# each column is the corresponding element given in terms of the integral basis. 

def elems_to_basis (x, B):
    return (elems_to_matrix_power(B,len(B))).inverse() * (elems_to_matrix_power (x,len(B)))

In [8]:
# From a m x n - matrix of lists (of length r), constructs a r x (mn) matrix that has those lists as columns: 
# The entry (i,j) corresponds to the ni + j column 
def matrix_of_matrix_lists (M, m, n):
    L = []
    for i in range(m):
        for j in range(n):
            L = L + [M[i][j]]
    M = Matrix(L).transpose()
    return(M)

In [9]:
# Compute a Z-basis for the ideal I in upper-triangular form. The columns are the elements wrt the integral basis B.
def IntegralBasisIdeal (I, B):
    Z_basis = I.basis()
    W = (((Matrix(ZZ, n, n, (elems_to_basis(Z_basis,B))).transpose())[: , ::-1].echelon_form())[: , ::-1])[::-1 , :].transpose()
    return W 

In [10]:
# Given an 'almost' orthogonal basis given as columns of a matrix M, 'project' a vector b on the lattice spanned by the 
# basis and subtract it from the original vector. We pick the number of vectors from the basis that results in the minimal 
# euclidean length of the result. 

def PseudoProject (M, b): 
    n = len(M.columns())
    if n == 0 :
        return b , float(norm(b))
    else :
        L = []
        x = 0 * M[:, 0]
        S = []
        for i in range(n):
            L = L + [floor((vector(M[: , i]).dot_product(vector(b))) / (vector(M[: , i]).dot_product(vector(M[: , i]))))]
        for m in range(n):
            for j in range(m) : 
                x = x + L[j] * M[:, j]
            S = S + [(vector(b) - vector(x), norm(vector(b) - vector(x)))]
        Ls = sorted (S, key = lambda x: x[1])
    
        return Ls[0][0], float(Ls[0][1])

In [11]:
#Given a matrix A and vector b, solve the system A*x = b in the integers. We use the HNF of A for this.  
# After obtaining a result b', we approximately project it into the lattice of kernel elements of the matrix, and then 
# substract the projection from b in order to obtain a solution of smaller norm. 

def SolveRightInteger(M,b): 
    MAux, Echel = M.transpose().echelon_form(transformation=True)
    faux = Echel.transpose() * (MAux.transpose()).solve_right(b)
    faux, nm = PseudoProject(Echel.transpose()[: , len(M.rows()) : ], faux)
    return faux

In [12]:
# The generators of the ideal as an O-module are given in ideal_gens. 

def IdealEqSpanCertificate(K, ideal_gens,B): 
    O = K.ring_of_integers()
    I = O.ideal(ideal_gens)
    
    Z_basis = I.basis()
    W = (((Matrix(ZZ, n, n, (elems_to_basis(Z_basis, B))).transpose())[: , ::-1].echelon_form())[: , ::-1])[::-1 , :].transpose()
    M = Matrix(K, len(ideal_gens), n)
    MB = [[[] for j in range(n)] for i in range(len(ideal_gens))]

    for i in range(len(ideal_gens)):
        for j in range(n):
            M[i, j] = (ideal_gens[i]) * B[j]
            MB[i][j] = (elems_to_basis ([M[i,j]], B)).list()

    Mland = Matrix(ZZ, matrix_of_matrix_lists(MB,len(ideal_gens), len(B)))
    faux = [() for k in range(n)]
    # We solve the system Mland * x = W k in the integers. 
    faux = [() for k in range(n)]
    for k in range(n):
        faux[k] = SolveRightInteger(Mland, (vector(W[:, k])))

    f = [[[[] for j in range(n)] for i in range(len(ideal_gens))] for k in range(n)]
    for k in range(n):
        for i in range(len(ideal_gens)):
            for j in range(n):
                f[k][i][j] = faux[k][n * i + j]
               
    g = [[[[] for k in range(n)] for j in range(n)] for i in range(len(ideal_gens))]
    for i in range(len(ideal_gens)):
        for j in range(n) :
            g[i][j] = (W.solve_right(vector(MB[i][j]))).list()
    
    w = [[] for i in range(n)]  
    v = [[] for i in range(len(ideal_gens))]
    
    for i in range(len(ideal_gens)):
        v[i] = (elems_to_basis ([ideal_gens[i]], B)).list()
    for j in range(n):
        w[j] = (W[: , j]).list()
        
    return Mland, f, g, v, w

In [13]:
#Given a string, this outputs the string corresponding to placing a ! mark in front of every [ 

def ExList (string) : 
    l = ''
    for x in string:
        if x == '[' :
            l = l + '!['
        else : 
            l = l + x
    return l

In [14]:
# Compute multiplication table for an order in a number field by giving the defining polynomial f and a list with the basis elements. 
def MultiplicationTable(f, B):
    n = len(B)
    K.<a> = NumberField(f)
    #Columns of this matrix are basis for Ok in terms of 1, a^2, ..., a^n. It is upper triangular. 
    A = [[[] for j in range(n)] for i in range(n)]
    for i in range(n):
        for j in range(n):
            A[i][j] = (elems_to_basis([B[i] * B[j]], B)).list()
    return A

In [219]:
R.<x> = PolynomialRing(QQ)
K.<a> = NumberField(x ^ 5 - 3 * x ^ 3 + 9 * x - 8 , 'a')
#a = K.gen()
B = [1, 1/2*a^4 + 1/2*a, 1/2*a^4 + 1/2*a^3 + 1/2*a^2, a^3, a^4]
MultiplicationTable(x ^ 5 - 3 * x ^ 3 + 9 * x - 8, B)

'![![![1, 0, 0, 0, 0], ![0, 1, 0, 0, 0], ![0, 0, 1, 0, 0], ![0, 0, 0, 1, 0], ![0, 0, 0, 0, 1]], ![![0, 1, 0, 0, 0], ![4, 3, -13, 10, 5], ![8, -2, -14, 10, 9], ![12, -27, 8, -4, 10], ![4, 15, -27, 19, 6]], ![![0, 0, 1, 0, 0], ![8, -2, -14, 10, 9], ![16, -12, -19, 13, 18], ![16, -28, -1, 2, 16], ![12, 5, -28, 18, 13]], ![![0, 0, 0, 1, 0], ![12, -27, 8, -4, 10], ![16, -28, -1, 2, 16], ![0, 16, -18, 9, 4], ![24, -54, 16, -8, 19]], ![![0, 0, 0, 0, 1], ![4, 15, -27, 19, 6], ![12, 5, -28, 18, 13], ![24, -54, 16, -8, 19], ![0, 48, -54, 35, 3]]]'

In [15]:
def IdealEqSpanCertificateLean(K, ideal_gens,B,name,ideal_name):
    Mland, f, g, v, w = IdealEqSpanCertificate(K, ideal_gens,B)
    print ("def " + name + ": IdealEqSpanCertificate O ℤ TT " + ideal_name + '\n '+ ExList(str(v)) + ' \n ' + ExList(str(w)) + ' where')
    print("""  T := T 
  heq := T_eq_TT
  hieq := rfl """)
    bt = ZZ(len((Mland[0 , :]).list()) / len(ideal_gens))
    M = [[[] for j in range(n)] for i in range(len(ideal_gens))]
    for i in range(len(ideal_gens)):
        for j in range(bt):
            M[i][j] = (Mland [: , bt * i + j]).list()
    print("  M :=" + ExList(str(M)))
    print("  hmulB := by decide")
    print("  f := " + ExList(str(f)))
    print("  g := " + ExList(str(g)))
    print("  hle1 := by decide")
    print("  hle2 := by decide")
    return w 

In [491]:
R.<x> = PolynomialRing(QQ)

K = NumberField(x^5 - 3*x^3 + 9*x - 8 , 'a')
n = K.degree()
a = K.gen()
B = [1, 1/2*a^4 + 1/2*a, 1/2*a^4 + 1/2*a^3 + 1/2*a^2, a^3, a^4]  # Basis for the ring of integers
O = K.ring_of_integers()
ideal_gens = [43, a + 11]  
I = O.ideal(ideal_gens)

IdealEqSpanCertificateLean(K, ideal_gens,B, 'A','I')

[43 38 14 41 22]
[ 0  1  0  0  0]
[ 0  0  1  0  0]
[ 0  0  0  1  0]
[ 0  0  0  0  1]
def A: IdealEqSpanCertificate O ℤ TT I
 ![![43, 0, 0, 0, 0], ![11, 2, 0, 0, -1]] 
 ![![43, 0, 0, 0, 0], ![38, 1, 0, 0, 0], ![14, 0, 1, 0, 0], ![41, 0, 0, 1, 0], ![22, 0, 0, 0, 1]] where
  T := T 
  heq := T_eq_TT
  hieq := rfl 
  M :=![![![43, 0, 0, 0, 0], ![0, 43, 0, 0, 0], ![0, 0, 43, 0, 0], ![0, 0, 0, 43, 0], ![0, 0, 0, 0, 43]], ![![11, 2, 0, 0, -1], ![4, 2, 1, 1, 4], ![4, -9, 11, 2, 5], ![0, 0, 0, 11, 1], ![8, -18, 0, 3, 20]]]
  hmulB := by decide
  f := ![![![730023, 381764, 192984, 195908, 782034], ![186362, -8366424, 6192, -6364, 0]], ![![673482, 352193, 178035, 180703, 721452], ![171892, -7718337, 5712, -5756, 0]], ![![234851, 122814, 62083, 63032, 251581], ![59943, -2691480, 1992, -2080, 0]], ![![696071, 364008, 184008, 186795, 745660], ![177684, -7977288, 5904, -6064, 0]], ![![373505, 195322, 98736, 100232, 400110], ![95329, -4280496, 3168, -3256, 0]]]
  g := ![![![1, 0, 0, 0, 0], ![-38, 43, 

In [16]:
#Given the O-generators a_1, ..., a_k of an ideal in a number field K with basis B and an element in this ideal, the function 
# computes elements x1, x2, .., x_k in the ring of integers such that x1 * a_1 + ... + x_k * a_k = elem. 

def IdealLift(K, B, ideal_gens, elem): 
    n = len(B)
    O = K.ring_of_integers()
    
    I = O.ideal(ideal_gens)
    
    M = Matrix(K, len(ideal_gens), n)
    MB = [[[] for j in range(n)] for i in range(len(ideal_gens))]
    
    for i in range(len(ideal_gens)):
        for j in range(n):
            M[i, j] = (ideal_gens[i]) * B[j]
            MB[i][j] = (elems_to_basis ([M[i,j]], B)).list()
    
    Mland = Matrix(ZZ, matrix_of_matrix_lists(MB,len(ideal_gens), n))
    faux = SolveRightInteger(Mland, (vector(elems_to_basis([elem], B))))
    #print(faux)
    flag = 0 
    for z in faux:
        if z not in ZZ:
            flag = 1
            print('Not inside the ideal')
            return 0 
    k = len(ideal_gens)
    x = [0 for j in range(k)]
    
    for j in range(k):
        
        for i in range(n):
            x[j] = x[j] + faux[n * j + i]* B[i]

    check = 0 
    for j in range(k):
        check = check + x[j] * ideal_gens[j]
    if check == elem:
        return x 
    else:
        print('Something went wrong.')
        

In [296]:
R.<x> = PolynomialRing(QQ)
K.<a> = NumberField(x^5 - 5*x^3 + 10*x - 4 , 'a')
ideal_gens = [11, a^2 + 3*a + 3] 
elem = 22 + 2 * a^2 + 6*a + 6 
B = [1, a, a^2, a^3, 1/2*a^4 - 1/2*a^2]
x = IdealLift(K, B, ideal_gens, elem)
print(x)

[-a^4 - 15*a^3 - 25*a^2 + 6*a + 44, 11*a^2 + 132*a - 152]


In [17]:
def IdealMultiplication (K, B, ideal_gens1, ideal_gens2):
    O = K.ring_of_integers()
    I1 = O.ideal(ideal_gens1)
    I2 = O.ideal(ideal_gens2)
    ideal_gens3 = (I1 * I2).gens_reduced() 
    M = [[[] for j in range(len(ideal_gens2))] for i in range(len(ideal_gens1))]
    Mcoord = [[[] for j in range(len(ideal_gens2))] for i in range(len(ideal_gens1))]
    for i in range(len(ideal_gens1)):
        for j in range(len(ideal_gens2)):
            M[i][j] = ideal_gens1[i] * ideal_gens2[j]    
            Mcoord[i][j] = (elems_to_basis([M[i][j]], B)).list()
    Maux = []
    for i in range(len(ideal_gens1)):
        Maux = Maux + M[i]
    f = [[[[] for i in range(len(ideal_gens2))] for j in range(len(ideal_gens1))] for k in range(len(ideal_gens3))]
    for i in range(len(ideal_gens3)): 
        faux = IdealLift(K, B, Maux, ideal_gens3[i])
        for j in range(len(ideal_gens1)):
            for k in range(len(ideal_gens2)):
                f[i][j][k] = elems_to_basis([faux[(len(ideal_gens2)) * j + k]], B).list() 
    g = [[[[] for i in range(len(ideal_gens3))] for j in range(len(ideal_gens2))] for k in range(len(ideal_gens1))]
    for i in range(len(ideal_gens1)):
        for j in range(len(ideal_gens2)) : 
            for k in range(len(ideal_gens3)): 
                g[i][j][k] =  elems_to_basis([IdealLift(K, B, ideal_gens3, M[i][j])[k]], B).list()
    return ideal_gens3, Mcoord, f, g
            

In [18]:
def IdealMulEqCertificateLean (nameC, K, B, ideal_gens1, ideal_gens2, proof1_name, proof2_name, ideal1_name,ideal2_name) :
    ideal_gens3, Mcoord, f, g = IdealMultiplication(K,B,ideal_gens1, ideal_gens2)
    print(f"def {nameC} : IdealMulEqCertificate O ℤ TT " + ideal1_name + ' ' + ideal2_name)
    ideal_gens1_coord = [ elems_to_basis([ideal_gens1[i]], B).list() for i in range(len(ideal_gens1))]
    ideal_gens2_coord = [ elems_to_basis([ideal_gens2[i]], B).list() for i in range(len(ideal_gens2))]
    ideal_gens3_coord = [ elems_to_basis([ideal_gens3[i]], B).list() for i in range(len(ideal_gens3))]
    print(' ', ExList(str(ideal_gens1_coord)), ExList(str(ideal_gens2_coord)))
    print(' ', ExList(str(ideal_gens3_coord)), 'where')
    print(f""" T := T
 heq := T_eq_TT
 hI1 := {proof1_name}
 hI2 := {proof2_name}""")
    print(' M := ', ExList(str(Mcoord)))
    print(' hmul := by decide')
    print(' f := ', ExList(str(f)))
    print(' g := ', ExList(str(g)))
    print(' hle1 := by decide')
    print(' hle2 := by decide')
    return ideal_gens3
    

In [19]:
def IteratedMulLean (K, B, ideal_gens, ideal_pows, proof1_name, proof2_name, ideal_name): 
    if len(ideal_gens)> 1:
        ideal_name_accum = ideal_name + '0'
    else : 
        ideal_name_accum = ideal_name 
    m = len(ideal_gens)
    ideal_gens_accum = ideal_gens[0]
    cont = 0 
    proof_name = proof1_name
    for i in range(m):
        if i == 0:
            for j in range(ideal_pows[i] - 1) :
                
                if len(ideal_gens)> 1:
                    ideal_gens_accum = IdealMulEqCertificateLean('Mul' + str(ideal_name) + str(cont),K, B, ideal_gens_accum , ideal_gens[i], proof_name, proof2_name, '(' + ideal_name_accum + ')', str(ideal_name) + '0')
                    ideal_name_accum = ideal_name_accum + '*' + str(ideal_name) + '0'
                else : 
                    ideal_gens_accum = IdealMulEqCertificateLean('Mul' + str(ideal_name) + str(cont),K, B, ideal_gens_accum , ideal_gens[i], proof_name, proof2_name, '(' + ideal_name_accum + ')', str(ideal_name) )
                    ideal_name_accum = ideal_name_accum + '*' + str(ideal_name) 
                print('')
                proof_name = "ideal_eq_mul_of_IdealMulEqCertificate O ℤ TT _ _ _ _ _ " + 'Mul' + str(ideal_name) + str(cont)
                cont = cont + 1
        else : 
            for j in range(ideal_pows[i]) :
                ideal_gens_accum = IdealMulEqCertificateLean('Mul' + str(ideal_name) + str(cont), K, B, ideal_gens_accum, ideal_gens[i], proof_name, proof2_name, '(' + ideal_name_accum + ')', str(ideal_name) + str(i))
                ideal_name_accum = ideal_name_accum + '*' + str(ideal_name) + str(i)   
                print('')
                proof_name = "ideal_eq_mul_of_IdealMulEqCertificate O ℤ TT _ _ _ _ _ " + 'Mul' + str(ideal_name) + str(cont)
                cont = cont + 1
    return ideal_gens_accum 

In [20]:
# Certificate of membership using a Z-basis. Input is the generators of the ideal as an O-module and the coordinates of the 
# element. 
def IdealMemZLean (K, B,ideal_gens, mem_coords, name_lemma, name_ideal, cert_name_eq) :
    Mland, f, g, v, w = IdealEqSpanCertificate(K, ideal_gens,B)
    g = Matrix(w).transpose().solve_right(vector(mem_coords))
    flag = 0
    for x in g:
        if x not in ZZ:
            print('Element is not a member of the ideal')
            flag = 1
    if flag == 0 :
        print(f"def {name_lemma} : IdealMemCertificate O ℤ B {name_ideal}")
        print(' ',ExList(str(w)), ExList(str(mem_coords)), 'where')
        print(f' hieq := ideal_eq_of_IdealEqSpanCertificate _ _ {cert_name_eq}')
        print(' g :=', ExList(str(g.list())))
        print(' hmem := by decide')

In [21]:
def IsOrderOf (p, name): 
    print (f'def {name} : IsOrderOf ({primitive_root(p)} : ZMod {p}) {p-1} where')
    F = factor(p-1)
    m = len(F)
    print(' m :=', m)
    P = [x[0] for x in F]
    e = [x[1] for x in F]
    print(' P :=', ExList(str(P)))
    print(' e :=', ExList(str(e)))
    print(""" hP := by decide
 hm := by rfl
 hid := by decide
 hnid := by decide""")

In [22]:
# We are brute forcing the computation of this discrete logarithm. If we have a map to Z/nZ, we can use more efficient 
# methods. Find log(x) with basis z. 

def DiscreteLog (O, I, x, z): 
    R = QuotientRing(O , I, 'y')
    for i in range(I.norm()):
        if R(x) == R(z) ^ i :
            return i , Mod(z ^ i, I.norm())

This part is to compute the ideals and the submatrix that makes everything work. 

In [91]:
# Output: 
# The matrix proving that x is not a p-power times a unit. Which proves that if J ^ p = (x) implies that J is not principal. 
# The list of prime ideals of degree 1 that correspond to the rows of the matrix. 
# The list of elements corresponding to the columns of the matrix: generators for the unit group mod torsion, the torsion unit (if p divides 
# the torsion order) and x. 
# A flag indicating if p divides torsion order (flag = 1) or not (flag = 0).

#The order of the ideals is in such a way that the submatrix obtained by eliminating the last column and the last row is of full-rank. 

def MatrixPrimes (K, O, p, x, T) :
    # L is the list of elements representing the columns. 
    L = []
    U = K.unit_group().gens()

    flag_dvdtor = 0 
    
    for i in range(len(U)):
        if i == 0 :
            if order(K.unit_group().gens()[0]) % p == 0: # If p divides torsion order, we include it in the columns 
                L = L + [K.unit_group().gens()[0]]
                flag_dvdtor = 1
        else : 
            L = L + [K.unit_group().gens()[i]]
            
    if flag_dvdtor == 1 : 
        # Move the first column to the last 
        L = L[1:] + [L[0]]+ [x]
    else : 
        L = L + [x]
    
    Ql = []
    A = []
    cont = 0 
    for q in primes(T + 1):
        if (q - 1) % p == 0 : 
            z = primitive_root(q)
            F = factor(O.ideal(q))
            for Q in F:
                if Q[0].norm() == q :
                    B = A + [[DiscreteLog(O, Q[0], L[j], z)[0] % p for j in range(len(L))]]
                    if Matrix(B).change_ring(GF(p)).rank() > cont :
                        Ql = Ql + [Q]
                        cont = cont + 1
                        A = B 
        if cont == len(L) :
            pivots = Matrix(GF(p), A)[:, :-1].transpose().pivots()
            A_initial = [A[i] for i in pivots]
            A_last = [A[i] for i in range(len(L)) if i not in pivots]
            Ql_initial = [Ql[i][0] for i in pivots]
            Ql_last = [Ql[i][0] for i in range(len(L)) if i not in pivots]
            A = A_initial + A_last
            Ql = Ql_initial + Ql_last
            
            return Matrix(A), Ql, L, flag_dvdtor
            
    return 0 

In [24]:
# Note that this assumes the prime is of inertia degree one. 

def DiscreteLogCLean (K, B, name, ideal_gens, hcard, hprim_root, p, x, x_name, name_ideal, cert_name_eq):
    mem_coords = elems_to_basis([x], B).list()
    O = K.ring_of_integers()
    I = O.ideal(ideal_gens)
    q = I.norm()
    k, m = DiscreteLog(O, I, x, primitive_root(q))
    mem_coords[0] = mem_coords[0] - ZZ(m)
    IdealMemZLean(K, B, ideal_gens, mem_coords, name + 'Mem', name_ideal, cert_name_eq)
    print('')
    print(f'def {name}: DiscreteLogCertificate {hcard} {hprim_root} {p} {x_name} {k % p} where')
    print(' r :=', len(B))
    print(' hN := by infer_instance')
    print(' hpdvd := by decide')
    print(' B := B')
    print(' hone := B_one')
    print(' xcoord := ', ExList(str(elems_to_basis([x], B).list())))
    print(' hxeq :=  rfl')
    print(' m:=', m)
    print(' C :=', ExList(str(mem_coords)))
    print(' hCeq := by rfl')
    print(' hmem := mem_of_certificate O ℤ _ _ _ _ ' + name + 'Mem' )
    print(' k := ', k)
    print(' hpow := by decide')
    print(' heql := by decide')
    return k % p , m

In [25]:
# Given a list of prime ideals of degree 1, and a list of elements. Prove primitive roots and compute 
# certificates for the discrete log of those elements in O/I. 

#IdealEqSpanCertificateLean(K, ideal_gens,B,name,ideal_name):

def FindNzEntry (w) : 
    for i in range(len(w)):
        for j in range(len(w[0])):
            if w[i][j] != 0:
                return i, j

def DiscreteLogListLean  (K, B, list_ideal_gens, ideal_name, p, list_elems, name_elems):
    O = K.ring_of_integers()
    N = set([O.ideal(x).norm() for x in list_ideal_gens])
    L = zero_matrix(len(list_ideal_gens), len(list_elems)) # The matrix of logarithms
    M = zero_matrix(len(list_ideal_gens), len(list_elems)) # The matrix of values under the isomorphism O/I \to mod q. 
    for q in N: 
        print(f'instance hq{q} : Fact $ Nat.Prime {q} := by decide')

    print('') 
    for q in N: 
        IsOrderOf(q, 'R' + str(q))
        print('')

    for i in range(len(list_ideal_gens)): 
        gensc = [elems_to_basis([x], B).list() for x in list_ideal_gens[i]]
        print(f'def {ideal_name}{i} : Ideal O := Ideal.span (Set.range (fun i ↦ TT.basis.equivFun.symm (' + ExList(str(gensc)) + ' i)))')
    print('')
        
    for i in range(len(list_ideal_gens)): 
        w = IdealEqSpanCertificateLean(K, list_ideal_gens[i], B, 'A' + str(i), ideal_name + str(i))
        ik, jk = FindNzEntry(w) 
        print('')
        print(f'lemma N{i} : Nat.card (O ⧸ {ideal_name}{i}) = {O.ideal(list_ideal_gens[i]).norm()} := ')
        print(f" ideal_norm_eq_prod' B _ _ (by decide) {ik} {jk} (by decide) (ideal_eq_of_IdealEqSpanCertificate O ℤ A{i})")            
        print('')

    for i in range(len(list_elems)): 
        print(f'def {name_elems[i]} := TT.basis.equivFun.symm', ExList(str(elems_to_basis([list_elems[i]], B).list())))
        print('')

    for i in range(len(list_ideal_gens)):
        for j in range(len(list_elems)): 
            l, m = DiscreteLogCLean(K, B, 'Log' + str(i) + str(j), list_ideal_gens[i], 'N' + str(i), f'((orderOf_of_IsOrderOf R{O.ideal(list_ideal_gens[i]).norm()}) ▸ IsPrimitiveRoot.orderOf _)', p, list_elems[j], name_elems[j], ideal_name + str(i), 'A' + str(i))
            print('')
            L[i,j] = l 
            M[i,j] = m 
    return L, M  

In [26]:
# Write a proof that I ^ m = <mem_elem>. 

def IdealPowLean (B, lemma_name , ideal_gens, ideal_name, elem, elem_name, m): 
    gensc = [elems_to_basis([x], B).list() for x in ideal_gens]
    print(f'def {ideal_name} : Ideal O := Ideal.span (Set.range (fun i ↦ TT.basis.equivFun.symm (' + ExList(str(gensc)) + ' i)))')
    a = IteratedMulLean(K, B, [ideal_gens], [m], 'rfl', 'rfl' , ideal_name)
    if a[0] != elem : 
        print("The generators don't match. Equal generators must be given")
        print('Computed: ', a[0])
        print('Given: ', elem)
    else: 
        print(f"lemma {lemma_name} : {ideal_name} ^ {m} = ", "Ideal.span {" + elem_name + "} := by")
        print(f' simp only [pow_succ, pow_one, pow_zero, one_mul]')
        print(f' simp [ideal_eq_mul_of_IdealMulEqCertificate O ℤ TT _ _ _ _ _ Mul{ideal_name}{m - 2}, {elem_name}]')
    

In [27]:
# Produce a prime ideal that proves that p does not divide the torsion. 
def PrimePNeDvd (K, B, p): 
    O = K.ring_of_integers()
    for q in prime_range(1,100): 
        if (q - 1) % p != 0 :
            F = factor(O.ideal(q))
            for I in F:
                if I[0].norm() == q: 
                    return I[0], [x for x in I[0].gens_reduced()]
                    
#[elems_to_basis([I[0].gens_reduced()[i]], B).list() for i in range(len(I[0].gens_reduced()))]

In [28]:
# Produces a proof that p does not divide the torsion order. If the degree of the number field is odd, then it does so 
# with the proof 'finrank_proof', otherwise it looks for a prime ideal of norm prime q such that p does not divide q-1. 
def pNeDvdTorsionLean(K, B, finrank_proof, integral_closure_eq, ideal_name, p): 
    if K.degree() % 2 != 0 and finrank_proof != '' :
        print(f'lemma ne_dvd_torsion{p} : ¬{p} ∣ Nat.card ↥(CommGroup.torsion (↥O)ˣ) := by')
        print(f""" suffices ¬ {p} ∣ 2 from ?_
 · convert this 
   rw [{integral_closure_eq}, ← Fintype.card_eq_nat_card,
   ←  NumberField.Units.torsionOrder_eq_two_of_odd_finrank (by rw [{finrank_proof}] ; decide)]
   rfl
 · decide""")
    else : 
        I, gens = PrimePNeDvd (K, B, p)
        gensc = [elems_to_basis([gens[i]], B).list() for i in range(len(gens))]
        print(f'def {ideal_name} : Ideal O := Ideal.span (Set.range (fun i ↦ TT.basis.equivFun.symm (' + ExList(str(gensc)) + ' i)))')
        print('')
        IdealEqSpanCertificateLean(K, gens, B, f'A{ideal_name}', ideal_name)
        print('')
        print(f' lemma ne_dvd_torsion{p} : ¬{p} ∣ Nat.card ↥(CommGroup.torsion (↥O)ˣ) := by ')
        print(f"""  refine prime_not_dvd_torsion_of_not_dvd (by decide) {ideal_name} 
     (ideal_norm_eq_prod' B _ _ (by decide) 0 0 (by decide) (ideal_eq_of_IdealEqSpanCertificate O ℤ A{ideal_name}))
     (by decide) (by decide)""")
        

In [29]:
#Generate proofs that units (defeq to TT.basis.equivFun.symm ![]) are indeed units, by computing and multiplying with the 
# inverse. An alternative could be computing the norm of I/(zeta) to be equal to 1. But this requires n multiplications. 

def IsUnitInvLean (B, list_elems, list_names):
    list_elems_inv = [x ^ (-1) for x in list_elems]
    gensc = [elems_to_basis([x], B).list() for x in list_elems_inv]
    for i in range(len(list_elems)): 
        print(f'lemma isUnit_{list_names[i]} : IsUnit {list_names[i]} := by ')
        print(f' apply isUnit_of_mul_eq_one _ (TT.basis.equivFun.symm {ExList(str(gensc[i]))})')
        print(""" rw [← B_one_repr]
 refine table_mul_list_eq_mul TT.table TT.basis _ _ _ TT.basis_mul_basis ?_
 rw [← table_mul_eq_table_mul' _ _ T_eq_TT]
 decide""")
        print('')

In [53]:
def SturmSequencePseudo (p, q): 
    R.<X> = PolynomialRing(ZZ)
    P = [p, q]
    if gcd(p,q) != 1:
        print('Polynomials p and q are not coprime') 
    else: 
        e = []
        f = []
        Q = []
        deg = p.degree()
        i = 0 
        while deg > 0 and i < 30: 
            delta = P[i].degree() - P[i + 1].degree() + 1
            e = e + [abs(((P[i + 1]).list()[-1]) ^ delta)]
            paux = e[i] * P[i] 
            p_i2 = - paux % P[i + 1]
            Q = Q + [ZZ[X]((paux + p_i2) / P[i + 1])]
            f = f + [gcd(p_i2.list())]
            P = P + [p_i2 / f[i]]
            deg = p_i2.degree()
            #print(deg)
            i = i + 1
            #print(i)
        return e, f, [q.list() for q in Q], [p.list() for p in P]


In [31]:
#Takes a separable polynomial with integer coefficients, produces a Lean certificate to count the real roots. 

def SturmCertificateLean (p, name): 
    q = p.derivative()
    e, f, Q, P = SturmSequencePseudo(p,q)
    print(f"""def {name} : SturmBuilderOfList {P} {P[0]} {P[1]} where
  hlen := by decide
  h0 := by decide
  h1 := by decide
  hlast := by decide
  hdrop := by decide
  hmono := by
    dsimp
    intro i hic 
    have hi : i < {len(P) - 1} := by omega
    interval_cases i <;> (dsimp ; decide)
  e := {e}
  f := {f}
  epos := by decide
  fpos := by decide
  Q := {Q}
  hel := by decide
  hfl := by decide
  hQl := by decide
  hrem := by
    dsimp
    intro i hi
    have hi : i < {len(P) - 2} := by omega
    interval_cases i <;> (dsimp ; decide)""")
    return P
    

In [32]:
def RankUnitCertificateLean (f, proof_ofList, proof_AdjoinRoot, proof_integral_closure_eq, name_cert): 
    R.<X> = PolynomialRing(ZZ)
    F.<x> = PolynomialRing(RR)
    P = SturmCertificateLean(f, f'Sturm{name_cert}')
    k = len((F(f)).roots())
    r = (k + f.degree()) / 2 
    print('')
    print(f'def {name_cert} : RankUnitsCertificate O where')
    print(f'  f := {R(f)}')
    print(f"""  l := {R(f).list()}
  hl := {proof_ofList}
  hlz := by decide
  hz := by decide
  hAdj := {proof_AdjoinRoot}
  heq := {proof_integral_closure_eq}
  P := {P}
  SB := Sturm{name_cert}
  k := {k}
  r := {r}
  hr := by decide
  hreq := by decide """)

In [42]:
#With v a torsion unit. Write a proof that v ^ m = 1, where m is the order of v. 

def TorsionUnitProof (K, B, v, name): 
    m = K(v).multiplicative_order()
    print(f"""lemma v_pow_one : {name} ^ {m} = 1 := by
  rw [← B_one_repr]
  apply table_nPow_sq_table_eq_pow TT.table T TT.basis _ TT.basis_mul_basis T_eq_TT _ (by norm_num)
  decide""")
    return m

In [34]:
# primitive_root(p)
def CertificateNonPrincipalityNT (M, p, m, zeta_names, prime_ideal_name, ideal_name, elem_name, primes_q, name_rank_cert, ideal_pow_proof):
    u0 = ''
    for x in zeta_names : 
        if u0 == '':
            u0 = x
        else : 
            u0 = u0 + ', ' + x
    In = ''
    #print(list(M[0,:]))
    d = len(list(M[0,:])[0])
    for i in range(d):
        if In == '':
            In = prime_ideal_name+str(i)
        else : 
            In = In + ', ' + prime_ideal_name+str(i)
    M = Matrix(GF(p), M)
    print(f'def NPC{ideal_name} : NonPrincipalCertificateNDvdT {ideal_name} where ')
    print(f""" p := {p}
 hp := by decide
 r := {len(zeta_names)}
 huc := by 
  rw [units_finrank_of_RankUnitsCertificate {name_rank_cert}]
  decide
 u := ![{u0}]
 hu := fun i => 
  match i with """)
    for i in range(len(zeta_names)): 
        print(f'  | {i} => isUnit_{zeta_names[i]}')
    print(f' q := {ExList(str(primes_q))}')
    print(f""" hqP := by 
  intro i 
  match i with """)
    for i in range(len(primes_q)): 
        print(f'  | {i} => exact hq{primes_q[i]}')
    print(f' I := ![{In}]')
    print(f""" hcard := fun i =>
  match i with  """)
    for i in range(d): 
        print(f'  | {i} => N{i}')
    print(f' ζ := !{[primitive_root(q) for q in primes_q]}')
    print(f""" hr := fun i =>
  match i with """)
    for i in range(len(primes_q)): 
        print(f'  | {i} => ((orderOf_of_IsOrderOf R{primes_q[i]}) ▸ IsPrimitiveRoot.orderOf _)')
    print(f""" hdvd := by decide
 a := {elem_name}
 n := {m}
 hpdvd := by decide
 hJ := {ideal_pow_proof}
 hpndvdt := ne_dvd_torsion{p}
 M := {ExList(str([list(list(M[i,:])[0]) for i in range(d)]))}
 hM1 := by 
  intro j i hj 
  fin_cases i <;> interval_cases j """)
    for i in range(d): 
        for j in range(d - 1): 
            print(f'  · dsimp ; rw [eq_of_DiscreteLogCertificate Log{i}{j}] ; decide')
    print(f""" hM2 := by 
  intro i 
  match i with """)
    for i in range(d):
        print(f'  | {i} => exact Log{i}{d-1}')
    print(f""" hDl := by decide
 Minv := {ExList(str([list(list((M ^ (-1))[i,:])[0]) for i in range(d)]))}
 hInv := by decide
 N := {ExList(str([list(list((M[0 : d - 1, 0 : d - 1] ^ (-1))[i,:])[0]) for i in range(d - 1)]))}
 hNiv := by decide""")
    

In [48]:
def CertificateNonPrincipalityDVDT (M, p, m, zeta_names, v_name, order_v, prime_ideal_name, ideal_name, elem_name, primes_q, name_rank_cert, ideal_pow_proof):
    u0 = ''
    for x in zeta_names : 
        if u0 == '':
            u0 = x
        else : 
            u0 = u0 + ', ' + x
    In = ''
    #print(list(M[0,:]))
    d = len(list(M[0,:])[0])
    for i in range(d):
        if In == '':
            In = prime_ideal_name+str(i)
        else : 
            In = In + ', ' + prime_ideal_name+str(i)
    M = Matrix(GF(p), M)
    print(f'def NPC{ideal_name} : NonPrincipalCertificateDvdT {ideal_name} where ')
    print(f""" p := {p}
 hp := by decide
 r := {len(zeta_names)}
 huc := by 
  rw [units_finrank_of_RankUnitsCertificate {name_rank_cert}]
  decide
 u := ![{u0}]
 hu := fun i => 
  match i with """)
    for i in range(len(zeta_names)): 
        print(f'  | {i} => isUnit_{zeta_names[i]}')
    print(f' v := {v_name}')
    print(f' m := {order_v}')
    print(f' hm := by norm_num')
    print(f' hmv := v_pow_one')
    print(f' q := {ExList(str(primes_q))}')
    print(f""" hqP := by 
  intro i 
  match i with """)
    for i in range(len(primes_q)): 
        print(f'  | {i} => exact hq{primes_q[i]}')
    print(f' I := ![{In}]')
    print(f""" hcard := fun i =>
  match i with  """)
    for i in range(d): 
        print(f'  | {i} => N{i}')
    print(f' ζ := !{[primitive_root(q) for q in primes_q]}')
    print(f""" hr := fun i =>
  match i with """)
    for i in range(len(primes_q)): 
        print(f'  | {i} => ((orderOf_of_IsOrderOf R{primes_q[i]}) ▸ IsPrimitiveRoot.orderOf _)')
    print(f""" hdvd := by decide
 a := {elem_name}
 n := {m}
 hpdvd := by decide
 hJ := {ideal_pow_proof}
 M := {ExList(str([list(list(M[i,:])[0]) for i in range(d)]))}
 hM1 := by 
  intro j i hj 
  fin_cases i <;> interval_cases j """)
    for i in range(d): 
        for j in range(d - 1-1): 
            print(f'  · dsimp ; rw [eq_of_DiscreteLogCertificate Log{i}{j}] ; decide')
    print(f""" hM2 := by 
  intro j  
  fin_cases j """)
    for i in range(d): 
        print(f'  · dsimp ; rw [eq_of_DiscreteLogCertificate Log{i}{d-1-1}] ; decide')
    print(f""" hM3 := by 
  intro i 
  match i with """)
    for i in range(d):
        print(f'  | {i} => exact Log{i}{d-1}')
    print(f""" hDl := by decide
 Minv := {ExList(str([list(list((M ^ (-1))[i,:])[0]) for i in range(d)]))}
 hInv := by decide
 N := {ExList(str([list(list((M[0 : d - 1, 0 : d - 1] ^ (-1))[i,:])[0]) for i in range(d - 1)]))}
 hNiv := by decide""")

In [54]:
R.<x> = PolynomialRing(QQ)
f = x^5 - 3*x^3 + 9*x - 8
K.<a> = NumberField(f, 'a')
O = K.ring_of_integers()
Cl = K.class_group('pari')
n = K.degree()
a = K.gen()
B = [1, 1/2*a^4 + 1/2*a, 1/2*a^4 + 1/2*a^3 + 1/2*a^2, a^3, a^4]
ideal_gens = [2, a]
J = O.ideal(ideal_gens)
m = Cl(J).order()
x = (J ^ m).gens_reduced()[0]
p = m.prime_divisors()[0]
A , Q , elems , flag = MatrixPrimes (K, O, p, x, 100)
elems = [K(r) for r in elems]
Ql = [list(Q[i].gens()) for i in range(len(Q))]
Qlq = [Ql[i][0] for i in range(len(Ql))]
names = ['zeta' + str(i + 1) for i in range(len(Q) - 1)] + ['alpha']

L, M = DiscreteLogListLean(K, B, Ql, 'I', p, elems, names)
IdealPowLean(B, 'J' + str(m), ideal_gens, 'J', x, 'alpha', m)
IsUnitInvLean(B, elems[:-1],names[:-1])
print("""

------------------------------------------------------------------------------------------------------
INCLUDE THESE AFTER GIVING NUMBER FIELD INSTANCE AND PROOFS OF RANK AND EQUALITY OF INTEGRAL CLOSURE.
-----------------------------------------------------------------------------------------------------


instance : Module.Finite ℤ (Additive ((↥O)ˣ ⧸ CommGroup.torsion (↥O)ˣ)) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFiniteIntAdditiveQuotientUnitsRingOfIntegersSubgroupTorsion K

instance : Module.Free ℤ (Additive ((↥O)ˣ ⧸ CommGroup.torsion (↥O)ˣ)) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFreeIntAdditiveQuotientUnitsRingOfIntegersSubgroupTorsion K

instance :  Fintype ↥(CommGroup.torsion (↥O)ˣ) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFintypeSubtypeUnitsRingOfIntegersMemSubgroupTorsion K

instance : IsCyclic ↥(CommGroup.torsion (↥O)ˣ) := by
  rw [O_integral_closure]
  exact NumberField.Units.instIsCyclicSubtypeUnitsRingOfIntegersMemSubgroupTorsion K

""")
RankUnitCertificateLean(R(f), 'ofList_poly', 'Adj_K', 'O_integral_closure', 'RC')
print('')
if flag == 0:
    pNeDvdTorsionLean(K, B, 'K_finrank' , 'O_integral_closure', 'IC', p)
    print('')
    CertificateNonPrincipalityNT (L, p, m, names[:-1], 'I', 'J', 'alpha', Qlq, 'RC', 'J' + str(m))
    print("""
theorem J_not_principal : ¬ ∃ b, J = Ideal.span {b} :=
  not_principal_of_NonPrincipalCertificateNDvdT NPCJ """)
else : 
    'p does divide torsion!'


instance hq43 : Fact $ Nat.Prime 43 := by decide
instance hq67 : Fact $ Nat.Prime 67 := by decide
instance hq19 : Fact $ Nat.Prime 19 := by decide

def R43 : IsOrderOf (3 : ZMod 43) 42 where
 m := 3
 P := ![2, 3, 7]
 e := ![1, 1, 1]
 hP := by decide
 hm := by rfl
 hid := by decide
 hnid := by decide

def R67 : IsOrderOf (2 : ZMod 67) 66 where
 m := 3
 P := ![2, 3, 11]
 e := ![1, 1, 1]
 hP := by decide
 hm := by rfl
 hid := by decide
 hnid := by decide

def R19 : IsOrderOf (2 : ZMod 19) 18 where
 m := 2
 P := ![2, 3]
 e := ![1, 2]
 hP := by decide
 hm := by rfl
 hid := by decide
 hnid := by decide

def I0 : Ideal O := Ideal.span (Set.range (fun i ↦ TT.basis.equivFun.symm (![![19, 0, 0, 0, 0], ![8, 2, 0, 0, -1]] i)))
def I1 : Ideal O := Ideal.span (Set.range (fun i ↦ TT.basis.equivFun.symm (![![43, 0, 0, 0, 0], ![-32, 2, 0, 0, -1]] i)))
def I2 : Ideal O := Ideal.span (Set.range (fun i ↦ TT.basis.equivFun.symm (![![67, 0, 0, 0, 0], ![21, 2, 0, 0, -1]] i)))

def A0: IdealEqSpanCertificate 

In [95]:
import re
R.<x> = PolynomialRing(QQ)
f = x^5 - 90*x^3 - 240*x^2 + 2160*x + 10368
PR.<X> = PolynomialRing(QQ)
K.<a> = NumberField(f, 'a')
O = K.ring_of_integers()
Cl = K.class_group('pari')
n = K.degree()
a = K.gen()
B = [1, a, 1/2*a^2, 1/12*a^3 - 1/2*a, 1/144*a^4 - 1/8*a^2 + 1/3*a]
ideal_gens = Cl.gens()[0].ideal().gens_reduced()
J = O.ideal(ideal_gens)
m = Cl(J).order()
x = (J ^ m).gens_reduced()[0]
p = m.prime_divisors()[0]
A , Q , elems , flag = MatrixPrimes (K, O, p, x, 200)
elems = [K(r) for r in elems]
Ql = [list(Q[i].gens()) for i in range(len(Q))]
Qlq = [Ql[i][0] for i in range(len(Ql))]
if flag == 0 : 
    names = ['zeta' + str(i + 1) for i in range(len(Q) - 1)] + ['alpha']
else : 
    names = ['zeta' + str(i + 1) for i in range(len(Q) - 2)] + ['v'] + ['alpha']

A = MultiplicationTable(f,B)

Am = re.sub(r'!(?=\[(\d|-\d))', '', ExList(str(A)))


print(f"""
--------------------------------------------
-- EXAMPLE
--------------------------------------------

/- Number field `K(α)` with with α root of polynomial `{PR(f)}`.
Class number `{m}`, generated by class of ideal `J = ({ideal_gens[0]}, {ideal_gens[1]})`. We have `J ^ {p} = (α)`. -/

/- Ring of integers with basis `{B}` -/

open BigOperators Classical Matrix Polynomial

noncomputable def P : ℤ[X] := {PR(f)} 

def K := AdjoinRoot (map (algebraMap ℤ ℚ) P)

lemma P_irreducible : Irreducible P := sorry
lemma P_degree : P.natDegree = 5 := sorry
lemma P_monic : Monic P := sorry

--local notation "K" =>  AdjoinRoot (map (algebraMap ℤ ℚ) ({PR(f)} ))

noncomputable instance : CommRing K := by
  unfold K
  infer_instance

noncomputable instance : Algebra ℚ K := by
  unfold K
  infer_instance

noncomputable def Adj_K : IsAdjoinRoot K (map (algebraMap ℤ ℚ) ({PR(f)} )) := by
  unfold K
  exact AdjoinRoot.isAdjoinRoot (Polynomial.map (algebraMap ℤ ℚ) ({PR(f)}))


lemma ofList_poly : ofList {f.list()} = {PR(f)}   := by
  simp ; ring


noncomputable def O : Subalgebra ℤ K := sorry

noncomputable def TT : TimesTable (Fin 5) ℤ O where
    basis := sorry
    table := {ExList(str(A))}
    basis_mul_basis := sorry 

noncomputable def B := TT.basis

lemma B_one : TT.basis 0 = 1 := sorry

lemma B_one_repr : TT.basis.equivFun.symm ![1, 0, 0, 0, 0] = 1 := sorry

def T : Fin 5 → Fin 5 → List ℤ :=
    {Am}

lemma T_eq_TT : ∀ (i j : Fin 5), T i j = List.ofFn (TT.table i j) := by
  simp ; decide

""")

print("""
instance : IsDomain O := by
  haveI hirr : Fact $ Irreducible (map (algebraMap ℤ ℚ) P) :=
  {out := (Polynomial.Monic.irreducible_iff_irreducible_map_fraction_map (P_monic)).1 P_irreducible}
  letI hola : Field K := by
    unfold K
    exact AdjoinRoot.instField
  haveI : IsDomain K := by infer_instance
  refine Subalgebra.isDomain O

noncomputable section 
""")


L, M = DiscreteLogListLean(K, B, Ql, 'I', p, elems, names)
IdealPowLean(B, 'J' + str(m), ideal_gens, 'J', x, 'alpha', m)
print('')
IsUnitInvLean(B, elems[:-1],names[:-1])
if flag == 1 :
    order_v = TorsionUnitProof(K, B, elems[len(elems) - 2], names[len(names) - 2])
    
print("""

instance hirr : Fact $ Irreducible (map (algebraMap ℤ ℚ) P) where
  out :=  (Polynomial.Monic.irreducible_iff_irreducible_map_fraction_map (P_monic)).1 P_irreducible

noncomputable instance K_field : Field K := by
  unfold K
  infer_instance

instance K_numberField : NumberField K := by
  unfold K
  infer_instance

lemma K_finrank : Module.finrank ℚ K = 5 := by
  unfold K
  rw [Module.finrank_eq_card_basis (AdjoinRoot.powerBasisAux _), Polynomial.natDegree_map_eq_of_injective, P_degree]
  · simp
  · exact RingHom.injective_int (algebraMap ℤ ℚ)
  · exact Irreducible.ne_zero hirr.out

theorem O_integral_closure : O = integralClosure ℤ K := by sorry

instance : Module.Finite ℤ (Additive ((↥O)ˣ ⧸ CommGroup.torsion (↥O)ˣ)) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFiniteIntAdditiveQuotientUnitsRingOfIntegersSubgroupTorsion K

instance : Module.Free ℤ (Additive ((↥O)ˣ ⧸ CommGroup.torsion (↥O)ˣ)) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFreeIntAdditiveQuotientUnitsRingOfIntegersSubgroupTorsion K

instance :  Fintype ↥(CommGroup.torsion (↥O)ˣ) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFintypeSubtypeUnitsRingOfIntegersMemSubgroupTorsion K

instance : IsCyclic ↥(CommGroup.torsion (↥O)ˣ) := by
  rw [O_integral_closure]
  exact NumberField.Units.instIsCyclicSubtypeUnitsRingOfIntegersMemSubgroupTorsion K

""")
RankUnitCertificateLean(R(f), 'ofList_poly', 'Adj_K', 'O_integral_closure', 'RC')
print('')
if flag == 0:
    pNeDvdTorsionLean(K, B, 'K_finrank' , 'O_integral_closure', 'IC', p)
    print('')
    CertificateNonPrincipalityNT (L, p, m, names[:-1], 'I', 'J', 'alpha', Qlq, 'RC', 'J' + str(m))
    print("""
theorem J_not_principal : ¬ ∃ b, J = Ideal.span {b} :=
  not_principal_of_NonPrincipalCertificateNDvdT NPCJ """)
else : 
    print('')
    CertificateNonPrincipalityDVDT (L, p, m, names[:-2], names[len(names) - 2], order_v, 'I', 'J', 'alpha', Qlq, 'RC', 'J' + str(m))
    print("""
theorem J_not_principal : ¬ ∃ b, J = Ideal.span {b} :=
  not_principal_of_NonPrincipalCertificateDvdT NPCJ """)



--------------------------------------------
-- EXAMPLE
--------------------------------------------

/- Number field `K(α)` with with α root of polynomial `X^5 - 90*X^3 - 240*X^2 + 2160*X + 10368`.
Class number `5`, generated by class of ideal `J = (2, 1/72*a^4 + 1/12*a^3 - 3/4*a^2 - 35/6*a - 5)`. We have `J ^ 5 = (α)`. -/

/- Ring of integers with basis `[1, a, 1/2*a^2, 1/12*a^3 - 1/2*a, 1/144*a^4 - 1/8*a^2 + 1/3*a]` -/

open BigOperators Classical Matrix Polynomial

noncomputable def P : ℤ[X] := X^5 - 90*X^3 - 240*X^2 + 2160*X + 10368 

def K := AdjoinRoot (map (algebraMap ℤ ℚ) P)

lemma P_irreducible : Irreducible P := sorry
lemma P_degree : P.natDegree = 5 := sorry
lemma P_monic : Monic P := sorry

--local notation "K" =>  AdjoinRoot (map (algebraMap ℤ ℚ) (X^5 - 90*X^3 - 240*X^2 + 2160*X + 10368 ))

noncomputable instance : CommRing K := by
  unfold K
  infer_instance

noncomputable instance : Algebra ℚ K := by
  unfold K
  infer_instance

noncomputable def Adj_K : IsAdjoinRoot K

In [262]:
R.<x> = PolynomialRing(QQ)
f = x^5 + 5*x^3 - 30*x^2 - 96
PR.<X> = PolynomialRing(QQ)
K.<a> = NumberField(f, 'a')
Cl.gens()[0].ideal().gens_reduced()



(2, 1/16*a^4 - 1/4*a^3 + 5/16*a^2 - 17/8*a + 11/2)

In [268]:
import re
re.sub(r'!(?=\[(\d|-\d))', '', '![![![2,3,4]]]')
#re.sub(r'!(?=\[\d+\])', '', '![![![2,3,4]]]')

'![![[2,3,4]]]'

In [41]:
K(-1).multiplicative_order()

2

In [90]:
L
print(Matrix(GF(5), L)[:, :-1].transpose().pivots())
pivots = Matrix(L)[:, :-1].pivots()
Matrix(L)[:, :-1].transpose().pivots()

(0, 2)


(0, 1)

In [65]:
A

[[[1, 0, 0, 0, 0],
  [0, 1, 0, 0, 0],
  [0, 0, 1, 0, 0],
  [0, 0, 0, 1, 0],
  [0, 0, 0, 0, 1]],
 [[0, 1, 0, 0, 0],
  [0, 0, 1, 0, 0],
  [0, 0, 1, 2, 0],
  [-38, -14, -9, -28, 83],
  [-12, -5, -1, -8, 27]],
 [[0, 0, 1, 0, 0],
  [0, 0, 1, 2, 0],
  [-76, -28, -17, -54, 166],
  [68, -61, 146, 102, -83],
  [-20, -35, 39, 6, 65]],
 [[0, 0, 0, 1, 0],
  [-38, -14, -9, -28, 83],
  [68, -61, 146, 102, -83],
  [-1474, -442, -448, -1001, 3154],
  [-494, -197, -77, -308, 1094]],
 [[0, 0, 0, 0, 1],
  [-12, -5, -1, -8, 27],
  [-20, -35, 39, 6, 65],
  [-494, -197, -77, -308, 1094],
  [-188, -91, -5, -108, 429]]]